# 04 — Land Surface Predictor QC (FIXED)

Uses the adapted land set:
**DEM/Elevation, NDVI, LST Day, Distance from Sea**.

LST Night is intentionally excluded because it is not part of this Khulna implementation.
Slope/Aspect may remain in the repository but are not part of the default paper-style combinations.

This notebook does not resample training predictors.

In [ ]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import re
import numpy as np
import pandas as pd
import rasterio

PRED_ROOT = RAW_DIR / "predictors"

def parse_ym(name):
    stem = Path(name).stem
    for pat in [r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
                r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"]:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))
    return None

def rasters(folder):
    return sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")]) if folder.exists() else []

for name in ["DEM","NDVI","LST_Day","Distance_Sea"]:
    folder = PRED_ROOT / name
    print(name, "->", len(rasters(folder)), "rasters")

In [ ]:
rows = []
for name in ["DEM","NDVI","LST_Day","Distance_Sea","Slope","Aspect"]:
    folder = PRED_ROOT / name
    for p in rasters(folder):
        with rasterio.open(p) as src:
            a = src.read(1, masked=True)
            v = a.compressed().astype("float64")
            rows.append({
                "predictor":name, "file":p.name, "path":str(p),
                "year_month":parse_ym(p.name),
                "crs":str(src.crs), "width":src.width, "height":src.height,
                "res_x":src.res[0], "res_y":src.res[1], "nodata":src.nodata,
                "valid_pct":100*v.size/a.size if a.size else np.nan,
                "min":float(np.nanmin(v)) if v.size else np.nan,
                "max":float(np.nanmax(v)) if v.size else np.nan,
                "mean":float(np.nanmean(v)) if v.size else np.nan,
                "bounds":str(tuple(src.bounds)),
            })

qc = pd.DataFrame(rows)
display(qc.head(30))
qc.to_csv(PROCESSED_DIR / "land_predictor_native_qc.csv", index=False)

In [ ]:
# Dynamic monthly coverage: NDVI and LST_Day
for name in ["NDVI","LST_Day"]:
    files = rasters(PRED_ROOT/name)
    by_ym = {}
    for p in files:
        ym = parse_ym(p.name)
        if ym:
            if ym in by_ym:
                raise ValueError(f"Duplicate {name} raster for {ym}: {by_ym[ym]} and {p}")
            by_ym[ym] = p
    missing = [(y,m) for y in range(2017,2023) for m in range(1,13) if (y,m) not in by_ym]
    print(f"{name}: parsed={len(by_ym)}, missing={len(missing)}")
    if missing:
        print(missing)

# Static predictors: choose one canonical file explicitly/heuristically.
def choose_static(folder_name, preferred_names):
    folder = PRED_ROOT / folder_name
    files = rasters(folder)
    if not files:
        raise FileNotFoundError(f"No raster found for {folder_name}")
    lower = {p.name.lower():p for p in files}
    for name in preferred_names:
        if name.lower() in lower:
            return lower[name.lower()]
    # Prefer filenames that do not contain obvious intermediate terms.
    clean = [p for p in files if not any(x in p.stem.lower() for x in ["clip","tmp","temp","aligned","resampl"])]
    if len(clean) == 1:
        return clean[0]
    if len(files) == 1:
        return files[0]
    raise ValueError(
        f"Multiple ambiguous {folder_name} rasters found. Set a canonical file manually:\n" +
        "\n".join(str(p) for p in files)
    )

DEM_PATH = choose_static("DEM", ["Khulna_SRTM_DEM.tif","DEM.tif"])
DFS_PATH = choose_static("Distance_Sea", ["Distance_Sea.tif","distance_to_sea.tif"])

print("Canonical DEM:", DEM_PATH)
print("Canonical Distance-to-Sea:", DFS_PATH)

(PROCESSED_DIR / "canonical_predictors.txt").write_text(
    f"DEM={DEM_PATH}\nDistance_Sea={DFS_PATH}\n",
    encoding="utf-8"
)

In [ ]:
print("\n04 complete. No training raster has been resampled.")